In [ ]:
import os
import pandas as pd
from IPython.display import display

 Metrik target: 'cer', 'WER', 'BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4',
                'ROUGE-1-P', 'ROUGE-1-R', 'ROUGE-1-F', 'BertScore'
TARGET_METRIC = 'cer'

PERCENTILES = [0, 10, 25, 35, 50, 65, 75, 85, 95, 100]

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../../../'))
DATA_DIR     = os.path.join(PROJECT_ROOT, 'experiments/ZuCo')

EXPERIMENTS = [
    {
        'exp': '2',
        'label': 'Exp 2 — ZGW | Log-Mel + Char LSTM (v6_1) | REAL',
        'file': 'complete_metrics_ZuCo_ZGW_NR_log-mel_fast_test_predictions_6_1.csv',
    },
    {
        'exp': '2',
        'label': 'Exp 2 — ZGW | Log-Mel + Char LSTM (v6_1) | NOISE BASELINE',
        'file': 'complete_metrics_ZuCo_NOISE_BASELINE_ZGW_NR_log-mel_fast_test_predictions_6_1.csv',
    },
    {
        'exp': '4',
        'label': 'Exp 4 — ALL | Log-Mel + Char LSTM (v6_1) | REAL',
        'file': 'complete_metrics_ZuCo_all_NR_log-mel_fast_test_predictions_6_1.csv',
    },
    {
        'exp': '4',
        'label': 'Exp 4 — ALL | Log-Mel + Char LSTM (v6_1) | NOISE BASELINE',
        'file': 'complete_metrics_ZuCo_NOISE_BASELINE_all_NR_log-mel_fast_test_predictions_6_1.csv',
    },
    {
        'exp': '5',
        'label': 'Exp 5 — ZGW | Hilbert + Char LSTM (v6_1) | REAL',
        'file': 'complete_metrics_ZuCo_ZGW_NR_hilbert_fast_test_predictions_6_1.csv',
    },
    {
        'exp': '5',
        'label': 'Exp 5 — ZGW | Hilbert + Char LSTM (v6_1) | NOISE BASELINE',
        'file': 'complete_metrics_ZuCo_NOISE_BASELINE_ZGW_NR_hilbert_fast_test_predictions_6_1.csv',
    },
    {
        'exp': '6',
        'label': 'Exp 6 — ZGW | Hilbert + GPT-2 Frozen (v10_1) | REAL',
        'file': 'complete_metrics_ZuCo_ZGW_NR_hilbert_GPT2_fast_test_predictions_10_1.csv',
    },
    {
        'exp': '6',
        'label': 'Exp 6 — ZGW | Hilbert + GPT-2 Frozen (v10_1) | NOISE BASELINE',
        'file': 'complete_metrics_ZuCo_NOISE_BASELINE_ZGW_NR_hilbert_GPT2_fast_test_predictions_10_1.csv',
    },
    {
        'exp': '10',
        'label': 'Exp 10 — ZGW | Log-Mel + GPT-2 Frozen (v13_1) | REAL',
        'file': 'complete_metrics_ZuCo_ZGW_NR_logmel_GPT2frozen_test_predictions.csv',
    },
    {
        'exp': '10',
        'label': 'Exp 10 — ZGW | Log-Mel + GPT-2 Frozen (v13_1) | NOISE BASELINE',
        'file': 'complete_metrics_ZuCo_NOISE_BASELINE_ZGW_NR_logmel_GPT2frozen_test_predictions.csv',
    },
]

print(f"Analisis Persentil — Metrik: {TARGET_METRIC.upper()}")
print(f"Direktori: {DATA_DIR}\n")

for exp in EXPERIMENTS:
    filepath = os.path.join(DATA_DIR, exp['file'])

    print("=" * 110)
    print(f"  {exp['label']}")
    print(f"  File: {exp['file']}")
    print("=" * 110)

    if not os.path.exists(filepath):
        print(f"  File tidak ditemukan: {filepath}\n")
        continue

    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        print(f"  Gagal membaca file: {e}\n")
        continue

    req_cols = [TARGET_METRIC, 'sentence', 'prediction']
    missing  = [c for c in req_cols if c not in df.columns]
    if missing:
        print(f"  Kolom tidak ditemukan: {missing}\n")
        continue

    df_sorted = df.sort_values(by=TARGET_METRIC).reset_index(drop=True)
    n = len(df_sorted)

    if n == 0:
        print("  File CSV kosong.\n")
        continue

    rows = []
    for p in PERCENTILES:
        idx = int((p / 100.0) * (n - 1))
        row = df_sorted.iloc[idx]
        rows.append({
            'Persentil (%)':  p,
            TARGET_METRIC:    round(row[TARGET_METRIC], 4),
            'sentence':       row['sentence'],
            'prediction':     row['prediction'],
        })

    df_pct = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 120)
    pd.set_option('display.width', 300)
    display(df_pct)
    print()